# SmolVLA fine-tune with the encoder trained and a piece-position loss (`mycobot-tri-sort-c-400`), Run C

Run C is Run B's data with three changes, set in `config/probe_protocol.yaml` (`run_c_settings`)
before this run:

* **The vision encoder (lr 1e-5) and the connector train** with the action expert (1e-4); the
  language layers stay frozen: 198.1 M of 450 M parameters (Run B: 100 M, encoder frozen).
* **An auxiliary loss**: a linear head (184 320 -> 8, **frozen** at its ridge warm start, 65.7 mm
  median error) reads the x, y of the four pieces from the connector output of the three cameras;
  total loss = action loss + **λ = 0.3** x masked MSE of the normalised positions. The head never
  changes; the loss trains the encoder and connector to keep the pieces' positions readable. The
  policy never sees the positions.
* **Batch 30 = 6 x 5 accumulated**, 10 000 updates (warmup 500, cosine decay), fp16 mixed
  precision with float32 weights, seed 1000.

Dataset: Run B's 400 episodes / 71 059 frames byte for byte, plus `meta/aux_labels.npz`,
`aux_head_init.safetensors`, `aux_warm_start.json`, `aux_warm_start_index.npz` and `SHA256SUMS`,
checked before anything is installed. The driver stops if the dataset is not exactly 400 episodes
and 71 059 frames.

**Stop conditions** (the driver stops training and says *GUARD STOP*; the run is then not
resumed): the auxiliary loss above 1 on a step up to 50; more than 5 of the first 100 updates
skipped by the fp16 loss scaler; the scaler below 1024 after step 100; a checkpoint whose head is
not identical to `aux_head_init.safetensors`. The prep run of 25/10 (head learning rate 1e-3)
would have stopped at step 10 on the first one.

**Between the two sessions** (`run_c_midway_stop`): the probe and image-swap tests run on
`kept/005000`; the second session runs only if they do not say stop.

## Kaggle, step by step

1. **Notebook**: *Code* -> *New Notebook* -> *File* -> *Import Notebook* -> this file
   (a **new** notebook).
2. **Session options**: *Accelerator* **GPU T4 x2** (one is used), *Internet* **On**,
   *Add Input* -> *Your Work* -> *Datasets* -> `mycobot-tri-sort-c-400`, **only this one**.
3. **Train**: *Save Version* -> **Save & Run All (Commit)**. Check in the log:
   *DATASET PASS*, *400 episodes, 71059 frames*, *freeze=runC ... total 198.1 M*,
   *lambda 0.3, head frozen*, two optimizer groups (no *aux head* group), and from step 500 on
   *aux head frozen: identical to aux_head_init.safetensors True, weight sum -0.0800367*.
4. **Session 1** stops cleanly at `SESSION_HOURS` (10.25 h). At about 6.5 s per update (2.6 s
   for 2 x 6 in the prep run) that is about step 5 600, ~4 900 on a T4 16 % slower; `kept/005000` is in its output. Download
   `kept/005000/pretrained_model/` for the midway tests (if the session ended before 5 000: the
   newest `run/checkpoints/<step>/pretrained_model/`, probe_protocol.yaml `run_c_midway_checkpoint`).
5. **Session 2**, only after the midway tests: open the notebook, *Add Input* -> *Your Work* ->
   *Notebooks* -> **this notebook** (its latest version), *Save & Run All* again. The log must show
   *fp16 loss scaler restored*. About 8 h. Handover notes: SESSION_RESUME.md, "Passation : Run C session 2". It ends with `TRAINING COMPLETE` and the final head
   check.
6. **Download**: `run/checkpoints/010000/pretrained_model/` and `kept/<step>/pretrained_model/`
   (2 500, 5 000, 7 500), plus `train_summary.json` and `runc_final_head.json`.

GPU quota: about 10.5 h for session 1, about 8.5 h for session 2.

In [ ]:
# ---- 1. Configuration ---------------------------------------------------------
MODE = "train"
TRAIN_STEPS = 10000          # Run C: warmup 500, cosine decay over 10 000
TRAIN_BATCH_SIZE = 30        # = MICRO_BATCH x 5 accumulated
MICRO_BATCH = 6              # accumulation check passed at batch 6 on the T4 (19/10)
TRAIN_AMP = True             # fp16 mixed precision
FP32_WEIGHTS = True          # float32 master weights (needed for TRAIN_AMP on a T4)
FREEZE = "runC"              # vision + connector + expert trained, language layers frozen
LR_VISION = 1e-5             # connector and expert: LeRobot's 1e-4
AUX_LAMBDA = 0.3             # calibrated 25/10 (exact 0.272), probe_protocol.yaml run_c_settings
AUX_HEAD_FROZEN = True       # the head stays at its warm start
SAVE_FREQ = 500
KEEP_EVERY = 2500            # kept weights for the midway tests (5 000) and the gate
SESSION_HOURS = 10.25        # session 1 fits the 11 h 29 left on 25/10 with ~1 h margin; ~5 600 steps at 6.5 s
LOG_FREQ = 10                # the stop conditions read the logged steps
GUARDS = {"aux_loss_max": 1.0, "aux_loss_steps": 50, "skip_max": 5, "skip_steps": 100,
          "scale_min": 1024, "scale_after": 100}   # probe_protocol.yaml run_c_settings.stop_conditions
AUX_INIT_WEIGHT_SUM = -0.0800367

TORCH_BACKEND = "auto"
VIDEO_BACKEND = None
NUM_WORKERS = 4
SEED = 1000
EXPECT_EPISODES, EXPECT_FRAMES = 400, 71059   # the driver stops on any other dataset

import os
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
PLATFORM, INPUT_ROOT, WORKING = "kaggle", "/kaggle/input", "/kaggle/working"
LOCAL = "/tmp/smolvla"          # venv, model cache, dataset copy: not saved as output
VENV = LOCAL + "/venv"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(PLATFORM, INPUT_ROOT, WORKING)

# ---- Dataset check: meta/SHA256SUMS (before anything is installed) -------------
import hashlib
sums = sorted(Path(INPUT_ROOT).rglob("meta/SHA256SUMS"))
assert len(sums) == 1, f"expected one meta/SHA256SUMS under {INPUT_ROOT}, found {sums}: attach mycobot-tri-sort-c-400"
META = sums[0].parent
checked = {line.split()[1]: hashlib.sha256((META / line.split()[1]).read_bytes()).hexdigest() == line.split()[0]
           for line in sums[0].read_text().splitlines()}
print("DATASET", "PASS" if all(checked.values()) else "FAIL", checked)
assert all(checked.values()), "a Run C file does not match meta/SHA256SUMS: stop"

In [ ]:
# ---- 2. Python 3.12 environment with LeRobot 0.6.1 (about 3-5 min) ------------
import subprocess, sys
from pathlib import Path

CONSTRAINTS = """aiohappyeyeballs==2.7.1
aiohttp==3.14.4
aiosignal==1.4.0
anyio==4.15.1
av==15.1.0
certifi==2026.7.22
charset-normalizer==3.5.2
click==8.5.0
cloudpickle==3.1.2
cmake==4.1.3
datasets==4.8.5
dill==0.4.1
draccus==0.11.6
einops==0.8.2
Farama-Notifications==0.0.6
frozenlist==1.8.0
fsspec==2026.2.0
grpcio==1.84.0
gymnasium==1.4.0
h11==0.16.0
hf-xet==1.7.0
httpcore==1.0.9
httpx==0.28.1
huggingface_hub==1.33.0
idna==3.20
jsonlines==4.0.0
lerobot==0.6.1
mergedeep==1.3.4
mpmath==1.3.0
multidict==6.9.1
multiprocess==0.70.19
mypy_extensions==1.1.0
networkx==3.6.1
numpy==2.2.6
opencv-python-headless==4.13.0.92
packaging==25.0
pandas==2.3.3
propcache==0.5.4
protobuf==7.36.2
pyarrow==26.0.0
requests==2.34.2
safetensors==0.8.0
scipy==1.18.1
setuptools==78.1.0
sympy==1.14.0
termcolor==3.3.0
toml==0.10.2
tqdm==4.70.1
typing-inspect==0.9.0
typing_extensions==4.16.0
tzdata==2026.5
urllib3==2.8.0
xxhash==4.0.1
yarl==1.25.1
accelerate==1.15.0
transformers==5.5.4
tokenizers==0.22.2
num2words==0.5.14
"""

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

if not Path(VENV, "bin", "lerobot-train").exists():
    Path(LOCAL).mkdir(parents=True, exist_ok=True)
    Path(LOCAL, "constraints.txt").write_text(CONSTRAINTS)
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"{sys.executable} -m uv venv --python 3.12 {VENV}")
    sh(f"{sys.executable} -m uv pip install --python {VENV}/bin/python --torch-backend={TORCH_BACKEND} "
       f"-c {LOCAL}/constraints.txt 'lerobot[smolvla,training]==0.6.1'")
sh(f"{VENV}/bin/python -c \"import torch, lerobot; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
   f"'|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU', '| lerobot', lerobot.__version__)\"")
Path(WORKING).mkdir(parents=True, exist_ok=True)
sh(f"{sys.executable} -m uv pip freeze --python {VENV}/bin/python > {WORKING}/pip_freeze.txt")
sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv")

## 3. Driver
The same file as `training/smolvla_kaggle/smolvla_kaggle_driver.py` in the repository, tested there on CPU (benchmark, stop at the time limit, resume from an attached output).

In [ ]:
%%writefile /tmp/smolvla_kaggle_driver.py
#!/usr/bin/env python3
"""SmolVLA fine-tune of local/mycobot_tri_sort on a Kaggle (or Colab) GPU.

Two modes, read from the JSON config written by the notebook's first cell:
  benchmark  short runs over batch sizes and precisions; prints s/step,
             samples/s, data-loading time, peak GPU memory and projections.
  train      fresh run, or resumes automatically from the newest complete
             checkpoint found under the input root (a previous run's output
             attached as input).

Standard library only: lerobot-train is run from the venv as a subprocess.
Paths come from the config so the same file runs on Kaggle, Colab and in the
gazebo_to_lerobot container (CPU) where it was tested.
"""
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import threading
import time
from pathlib import Path

REPO_ID = 'local/mycobot_tri_sort'
CAMERA_RENAME = {
    'observation.images.top': 'observation.images.camera1',
    'observation.images.right': 'observation.images.camera2',
    'observation.images.left': 'observation.images.camera3',
}
# smolvla_base loads in bfloat16 (447 M of 450 M parameters, 97 M of the 100 M trained).
# A T4 has no bfloat16 tensor cores, and fp16 mixed precision cannot unscale bfloat16
# gradients ("_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for
# 'BFloat16', Kaggle 2026-10-12). Casting the policy to float32 right after LeRobot builds
# it keeps float32 master weights; with use_amp the matmuls then run in float16.
#
# Gradient accumulation (SMOLVLA_MICRO_BATCH > 0): lerobot-train still loads and preprocesses
# the whole --batch_size batch, and update_policy is replaced by one that runs forward and
# backward on micro-batches of that size, then clips and steps once. Steps, the LR schedule,
# checkpoints and logs all count whole batches, as without accumulation.
#
# The fp16 loss scaler: LeRobot saves neither it nor restores it, so after a resume its scale
# restarted at 65536 and the first updates could be skipped. The launcher writes its state to
# training_state/grad_scaler.json with every checkpoint (before LeRobot's own files, so the last
# of those, scheduler_state.json, still marks a complete checkpoint) and restores it on resume.
#
# Run C (design note D1-D6, 19/10/2026), with SMOLVLA_AUX_DIR set to the dataset's meta folder:
# a linear auxiliary head on the connector output of the three cameras predicts the x, y of the
# four pieces (labels from aux_labels.npz, looked up by the frame's dataset index, never seen by
# the policy); total loss = action loss + SMOLVLA_AUX_LAMBDA x masked MSE on normalised
# positions. The head is kept out of the policy module: saved as pretrained_model/
# aux_head.safetensors next to each checkpoint, started from aux_head_init.safetensors (ridge
# warm start), in its own optimizer group (SMOLVLA_LR_HEAD); SMOLVLA_LR_VISION puts the vision
# encoder in its own group too. Two preparation modes run the same data path without training:
# SMOLVLA_AUX_FEATURES (frozen connector features of training frames, for the warm start) and
# SMOLVLA_AUX_CALIBRATE (gradient norms of both losses on the vision encoder, for lambda).
# SMOLVLA_AUX_HEAD_FROZEN keeps the head at its warm start (no gradient, no optimizer group): with
# 184 320 inputs averaging 3.5, Adam's first step at lr 1e-3 moved the head's output by ~100
# normalised units and threw the fp16 scaler down to 2 (Kaggle T4, 25/10). The auxiliary loss still
# trains the encoder and connector through it; every checkpoint's head is checked against the start.
FP32_LAUNCHER = '''import copy
import json
import os
import sys
import tempfile
import time
from collections import Counter
from contextlib import nullcontext

from pathlib import Path

import torch
import lerobot.scripts.lerobot_train as lt

_make_policy = lt.make_policy
_save_checkpoint = lt.save_checkpoint
_load_training_state = lt.load_training_state
ACCELERATOR = None
SCALER_STATE = None
SCALER_FILE = "grad_scaler.json"


FREEZE = os.environ.get("SMOLVLA_FREEZE") or None
MICRO_BATCH = int(os.environ.get("SMOLVLA_MICRO_BATCH") or 0)
ACCUM_CHECK = os.environ.get("SMOLVLA_ACCUM_CHECK") or None
AUX_DIR = os.environ.get("SMOLVLA_AUX_DIR") or None
AUX_LAMBDA = float(os.environ.get("SMOLVLA_AUX_LAMBDA") or 0)
AUX_FEATURES = os.environ.get("SMOLVLA_AUX_FEATURES") or None
AUX_FEATURES_FRAMES = int(os.environ.get("SMOLVLA_AUX_FEATURES_FRAMES") or 0)
AUX_CALIBRATE = os.environ.get("SMOLVLA_AUX_CALIBRATE") or None
LR_VISION = float(os.environ.get("SMOLVLA_LR_VISION") or 0)
LR_HEAD = float(os.environ.get("SMOLVLA_LR_HEAD") or 1e-3)
AUX_HEAD_FROZEN = bool(os.environ.get("SMOLVLA_AUX_HEAD_FROZEN"))
AUX_HEAD_FILE = "aux_head.safetensors"
AUX_INIT_FILE = "aux_head_init.safetensors"
AUX = None


def group(name):
    if name.startswith("aux_head."):
        return "aux head"
    if "vlm_with_expert.vlm." not in name:
        return "expert and projections"
    if "vision_model" in name:
        return "vision"
    if "connector" in name:
        return "connector"
    return "language"


class AuxHead:
    """The auxiliary position head and its labels (see the comment above FP32_LAUNCHER)."""

    def __init__(self, policy, pretrained_path):
        import numpy as np
        from safetensors.torch import load_file
        labels = np.load(Path(AUX_DIR) / "aux_labels.npz")
        size = int(labels["index"].max()) + 1
        idx = torch.from_numpy(labels["index"].astype("int64"))
        self.xy = torch.zeros(size, labels["xy"].shape[1])
        self.mask = torch.zeros(size, labels["mask"].shape[1], dtype=torch.bool)
        self.xy[idx], self.mask[idx] = torch.from_numpy(labels["xy"]), torch.from_numpy(labels["mask"])
        self.mean, self.std = torch.from_numpy(labels["mean"]), torch.from_numpy(labels["std"])
        saved = Path(str(pretrained_path)) / AUX_HEAD_FILE
        source = saved if saved.is_file() else Path(AUX_DIR) / AUX_INIT_FILE
        state = load_file(str(source))
        device = next(policy.parameters()).device
        self.head = torch.nn.Linear(state["weight"].shape[1], state["weight"].shape[0]).to(device)
        self.head.load_state_dict(state)
        self.init = load_file(str(Path(AUX_DIR) / AUX_INIT_FILE))
        if AUX_HEAD_FROZEN:
            self.head.requires_grad_(False)
        self.features = []
        policy.model.vlm_with_expert.get_vlm_model().connector.register_forward_hook(
            lambda _m, _i, out: self.features.append(out))
        print(f"[fp32 launcher] aux head {tuple(state['weight'].shape)} from {source}, weight sum "
              f"{float(state['weight'].double().sum()):.6g}; labels for {int(self.mask.any(1).sum())} frames, "
              f"lambda {AUX_LAMBDA}{', head frozen' if AUX_HEAD_FROZEN else ''}", flush=True)

    def count(self, batch):
        return 2.0 * float(self.mask[batch["index"].long().view(-1).cpu()].sum())

    def error_sum(self, batch):
        """Sum of squared errors over the valid normalised coordinates of the forward just run."""
        x = torch.cat([f.flatten(1) for f in self.features], 1)
        self.features.clear()
        if x.shape[1] != self.head.in_features:
            raise SystemExit(f"aux head expects {self.head.in_features} features, the connector gave {x.shape[1]}")
        i = batch["index"].long().view(-1).cpu()
        target = ((self.xy[i] - self.mean) / self.std).to(x.device)
        valid = self.mask[i].repeat_interleave(2, 1).to(x.device)
        return (((self.head(x).float() - target) ** 2) * valid).sum()

    def save(self, pretrained_dir):
        from safetensors.torch import save_file
        Path(pretrained_dir).mkdir(parents=True, exist_ok=True)
        state = {k: v.detach().contiguous().cpu() for k, v in self.head.state_dict().items()}
        save_file(state, str(Path(pretrained_dir) / AUX_HEAD_FILE))
        if AUX_HEAD_FROZEN:
            same = state.keys() == self.init.keys() and all(torch.equal(state[k], self.init[k]) for k in state)
            print(f"[fp32 launcher] aux head frozen: identical to {AUX_INIT_FILE} {same}, weight sum "
                  f"{float(state['weight'].double().sum()):.6g}", flush=True)


def trained_params(policy):
    params = [p for p in policy.parameters() if p.requires_grad]
    return params + (list(AUX.head.parameters()) if AUX and not AUX_HEAD_FROZEN else [])


def make_policy_fp32(*args, **kwargs):
    global AUX
    policy = _make_policy(*args, **kwargs).float()
    if FREEZE == "runC":
        # Language layers and lm_head frozen; gradients still flow through them to the vision encoder.
        for name, p in policy.named_parameters():
            if group(name) == "language":
                p.requires_grad = False
    if AUX_DIR:
        AUX = AuxHead(policy, kwargs["cfg"].pretrained_path)
    if LR_VISION or AUX:
        named = [(n, p) for n, p in policy.named_parameters() if p.requires_grad]
        # Same order in every session: the optimizer state of a resumed run is matched by position.
        groups = [{"params": [p for n, p in named if group(n) != "vision"], "name": "connector, expert"},
                  {"params": [p for n, p in named if group(n) == "vision"], "name": "vision",
                   **({"lr": LR_VISION} if LR_VISION else {})}]
        if AUX and not AUX_HEAD_FROZEN:
            groups.append({"params": list(AUX.head.parameters()), "name": "aux head", "lr": LR_HEAD})
        policy.get_optim_params = lambda: groups
        print("[fp32 launcher] optimizer groups: " + "; ".join(
            f"{g['name']} {sum(p.numel() for p in g['params']) / 1e6:.1f} M lr {g.get('lr', 'base')}"
            for g in groups), flush=True)
    trained = Counter(str(p.dtype) for p in policy.parameters() if p.requires_grad)
    by_group = Counter()
    for name, p in policy.named_parameters():
        if p.requires_grad:
            by_group[group(name)] += p.numel()
    print(f"[fp32 launcher] policy cast to float32; freeze={FREEZE}; trainable tensors by dtype: {dict(trained)}; "
          f"trainable parameters by group (M): { {k: round(v / 1e6, 1) for k, v in sorted(by_group.items())} }, "
          f"total {sum(by_group.values()) / 1e6:.1f} M", flush=True)
    return policy


def split_batch(batch, size):
    n = len(batch["action"])
    for start in range(0, n, size):
        yield {k: v[start:start + size]
               if (torch.is_tensor(v) and v.dim() > 0 and len(v) == n) or (isinstance(v, list) and len(v) == n)
               else v for k, v in batch.items()}


def update_policy_accumulated(train_metrics, policy, batch, optimizer, grad_clip_norm, accelerator,
                              lr_scheduler=None, lock=None, sample_weighter=None, micro=None):
    if sample_weighter is not None or accelerator.num_processes > 1:
        raise SystemExit("gradient accumulation: one GPU and no sample weighting only")
    start_time = time.perf_counter()
    policy.train()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    chunks = list(split_batch(batch, micro or MICRO_BATCH or len(batch["action"])))
    # The auxiliary loss is a mean over the whole batch's valid coordinates, so each micro-batch's
    # squared errors are divided by that whole-batch count: their sum is then the whole batch's.
    aux_count = max(AUX.count(batch), 1.0) if AUX else 0.0
    aux_value = 0.0
    # SmolVLA's loss is a mean over the batch's unpadded action steps, so each micro-batch is
    # weighted by its share of them: the summed gradient is then the whole batch's.
    if "action_is_pad" in batch:
        valid = [float((~c["action_is_pad"]).sum()) for c in chunks]
    else:
        valid = [float(len(c["action"])) for c in chunks]
    total = sum(valid) or 1.0
    loss_value, outputs = 0.0, {}
    for chunk, v in zip(chunks, valid):
        if AUX:
            AUX.features.clear()
        with accelerator.autocast():
            loss, out = policy.forward(chunk)
            objective = loss * (v / total)
            if AUX:
                aux = AUX.error_sum(chunk) / aux_count
                objective = objective + AUX_LAMBDA * aux
                aux_value += aux.item()
        accelerator.backward(objective)
        loss_value += loss.item() * v / total
        for k, x in (out or {}).items():
            if isinstance(x, (int, float)):
                outputs[k] = outputs.get(k, 0.0) + x * v / total
    if AUX:
        outputs["aux_loss"] = aux_value
    if grad_clip_norm > 0:
        grad_norm = accelerator.clip_grad_norm_(trained_params(policy), grad_clip_norm)
    else:
        grad_norm = torch.nn.utils.clip_grad_norm_(trained_params(policy), float("inf"), error_if_nonfinite=False)
    with lock if lock is not None else nullcontext():
        optimizer.step()
    if getattr(optimizer, "step_was_skipped", False):
        print(f"[fp32 launcher] update skipped by the fp16 loss scaler (inf/nan gradients), "
              f"scale now {accelerator.scaler.get_scale() if accelerator.scaler else None}", flush=True)
    optimizer.zero_grad()
    if lr_scheduler is not None:
        lr_scheduler.step()
    if lt.has_method(accelerator.unwrap_model(policy, keep_fp32_wrapper=True), "update"):
        accelerator.unwrap_model(policy, keep_fp32_wrapper=True).update()
    train_metrics.loss = loss_value
    train_metrics.grad_norm = grad_norm.item()
    train_metrics.lr = optimizer.param_groups[0]["lr"]
    train_metrics.update_s = time.perf_counter() - start_time
    if torch.cuda.is_available():
        train_metrics.gpu_mem_gb = torch.cuda.max_memory_allocated() / (1024**3)
    if outputs:
        train_metrics.update_metrics(outputs)
    return train_metrics, outputs


class FixedDraws:
    """Replays the noise and flow-matching times of the first (whole-batch) pass to the later ones,
    micro-batch by micro-batch, so every pass of the accumulation check sees the same draws."""

    def __init__(self, model):
        self.model, self.noise, self.time = model, None, None
        self.sample_noise, self.sample_time = model.sample_noise, model.sample_time
        model.sample_noise, model.sample_time = self.draw_noise, self.draw_time

    def rewind(self):
        self.at_noise = self.at_time = 0

    def draw_noise(self, shape, device):
        if self.noise is None:
            self.noise = self.sample_noise(shape, device)
        out = self.noise[self.at_noise:self.at_noise + shape[0]]
        self.at_noise += shape[0]
        return out

    def draw_time(self, bsize, device):
        if self.time is None:
            self.time = self.sample_time(bsize, device)
        out = self.time[self.at_time:self.at_time + bsize]
        self.at_time += bsize
        return out


def to_cpu(x):
    if torch.is_tensor(x):
        return x.detach().to("cpu", copy=True)
    if isinstance(x, dict):
        return {k: to_cpu(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return type(x)(to_cpu(v) for v in x)
    return copy.deepcopy(x)


def rel(a, b, names):
    num = sum(float(((a[n] - b[n]).double() ** 2).sum()) for n in names)
    den = sum(float((b[n].double() ** 2).sum()) for n in names)
    return (num / den) ** 0.5 if den > 0 else float("nan")


CHECK_ROWS = []


def update_policy_checked(train_metrics, policy, batch, optimizer, grad_clip_norm, accelerator,
                          lr_scheduler=None, lock=None, sample_weighter=None):
    """Accumulation check (SMOLVLA_ACCUM_CHECK=<report .json>): from the same weights, optimizer,
    scheduler and loss-scaler state, the same batch and the same noise and times, one update on
    the whole batch (twice: the second gives the run-to-run floor), then one accumulated over
    micro-batches of MICRO_BATCH, which training continues from. Compares the gradients the
    optimizer receives (unscaled, after clipping), the pre-clip gradient norm, the loss and the
    weight update, overall and per parameter group."""
    inner = accelerator.unwrap_model(policy, keep_fp32_wrapper=True)
    params = {n: p for n, p in inner.named_parameters() if p.requires_grad}
    if AUX:
        params.update({f"aux_head.{n}": p for n, p in AUX.head.named_parameters()})
    w0 = {n: to_cpu(p) for n, p in params.items()}
    # The optimizer state (2 x the trained weights) goes to disk, not RAM: a CPU run of the check
    # would not fit in 12 GB with it. In /tmp, not next to the report: on Kaggle the report's
    # folder is the session output, and a check cut by an out-of-memory error left 1.5 GB there.
    opt0 = os.path.join(tempfile.gettempdir(), "accum_check_optimizer.pt")
    torch.save(optimizer.state_dict(), opt0)
    sched0 = to_cpu(lr_scheduler.state_dict()) if lr_scheduler else None
    scaler0 = copy.deepcopy(accelerator.scaler.state_dict()) if accelerator.scaler else None
    draws = getattr(inner.model, "_fixed_draws", None) or FixedDraws(inner.model)
    inner.model._fixed_draws = draws
    draws.noise = draws.time = None
    grads = {}
    hook = optimizer.optimizer.register_step_pre_hook(
        lambda opt, args, kwargs: grads.update({n: to_cpu(p.grad) for n, p in params.items() if p.grad is not None}))

    def restore():
        with torch.no_grad():
            for n, p in params.items():
                p.copy_(w0[n].to(p.device))
        optimizer.load_state_dict(torch.load(opt0, map_location="cpu", weights_only=False))
        if lr_scheduler:
            lr_scheduler.load_state_dict(copy.deepcopy(sched0))
        if scaler0:
            accelerator.scaler.load_state_dict(copy.deepcopy(scaler0))

    def one(update):
        grads.clear()
        draws.rewind()
        update(train_metrics, policy, batch, optimizer, grad_clip_norm, accelerator=accelerator,
               lr_scheduler=lr_scheduler, lock=lock)
        return {"loss": train_metrics.loss.val, "grad_norm": train_metrics.grad_norm.val, "skipped": not grads,
                "grads": dict(grads), "delta": {n: to_cpu(p) - w0[n] for n, p in params.items()}}

    by_group = {}
    for n in params:
        by_group.setdefault(group(n), []).append(n)

    def compare(x, ref):
        if x["skipped"] or ref["skipped"]:
            return {"skipped": True}
        out = {"loss_rel": abs(x["loss"] - ref["loss"]) / abs(ref["loss"]),
               "grad_norm_rel": abs(x["grad_norm"] - ref["grad_norm"]) / ref["grad_norm"],
               "grads_rel": rel(x["grads"], ref["grads"], list(ref["grads"])),
               "update_rel": rel(x["delta"], ref["delta"], list(params))}
        for g, names in sorted(by_group.items()):
            out[f"grads_rel_{g}"] = rel(x["grads"], ref["grads"], [n for n in names if n in ref["grads"]])
            out[f"update_rel_{g}"] = rel(x["delta"], ref["delta"], names)
        return out

    # Without the head, LeRobot's own update is the reference; with it, the same update as one chunk.
    reference = _update_policy if AUX is None else (
        lambda *a, **k: update_policy_accumulated(*a, micro=len(batch["action"]), **k))
    try:
        whole = one(reference)
        restore()
        floor = compare(one(reference), whole)
        restore()
        accumulated = one(update_policy_accumulated)
    finally:
        hook.remove()
        os.remove(opt0)
    row = {"step": len(CHECK_ROWS) + 1, "batch": len(batch["action"]), "micro_batch": MICRO_BATCH,
           "loss": whole["loss"], "grad_norm": whole["grad_norm"],
           "missing_grads": sorted(set(params) - set(whole["grads"]))[:5] if not whole["skipped"] else [],
           "whole_again_vs_whole": floor, "accumulated_vs_whole": compare(accumulated, whole)}
    CHECK_ROWS.append(row)
    print("[accum check] " + json.dumps(row), flush=True)
    with open(ACCUM_CHECK, "w") as f:
        json.dump(CHECK_ROWS, f, indent=1)
    return train_metrics, {}


def save_checkpoint_with_scaler(checkpoint_dir, *args, **kwargs):
    if AUX:
        AUX.save(Path(checkpoint_dir) / "pretrained_model")
    scaler = ACCELERATOR.scaler if ACCELERATOR is not None else None
    if scaler is not None and scaler.is_enabled():
        state_dir = Path(checkpoint_dir) / "training_state"
        state_dir.mkdir(parents=True, exist_ok=True)
        (state_dir / SCALER_FILE).write_text(json.dumps(scaler.state_dict()))
    return _save_checkpoint(checkpoint_dir, *args, **kwargs)


def load_training_state_with_scaler(checkpoint_dir, *args, **kwargs):
    global SCALER_STATE
    out = _load_training_state(checkpoint_dir, *args, **kwargs)
    saved = Path(checkpoint_dir) / "training_state" / SCALER_FILE
    if saved.is_file():
        SCALER_STATE = json.loads(saved.read_text())
        print(f"[fp32 launcher] fp16 loss scaler state read from the checkpoint: {SCALER_STATE}", flush=True)
    else:
        print(f"[fp32 launcher] no {SCALER_FILE} in the checkpoint: the fp16 loss scaler starts afresh", flush=True)
    return out


def with_scaler(update):
    def wrapped(*args, accelerator, **kwargs):
        global ACCELERATOR, SCALER_STATE
        ACCELERATOR = accelerator
        if SCALER_STATE is not None:
            if accelerator.scaler is not None and accelerator.scaler.is_enabled():
                accelerator.scaler.load_state_dict(SCALER_STATE)
                print(f"[fp32 launcher] fp16 loss scaler restored: scale {accelerator.scaler.get_scale()}", flush=True)
            SCALER_STATE = None
        return update(*args, accelerator=accelerator, **kwargs)
    return wrapped


FEATURE_ROWS = {"index": [], "x": []}


def update_policy_features(train_metrics, policy, batch, optimizer, grad_clip_norm, accelerator, **kwargs):
    """No training: the frozen connector output of the three cameras for this batch's frames."""
    import numpy as np
    inner = accelerator.unwrap_model(policy, keep_fp32_wrapper=True)
    inner.eval()
    with torch.no_grad(), accelerator.autocast():
        images, _ = inner.prepare_images(batch)
        x = torch.cat([inner.model.vlm_with_expert.embed_image(img).flatten(1) for img in images], 1)
    FEATURE_ROWS["index"].append(batch["index"].view(-1).cpu().numpy())
    FEATURE_ROWS["x"].append(x.half().cpu().numpy())
    done = sum(len(i) for i in FEATURE_ROWS["index"])
    print(f"[aux features] {done} frames", flush=True)
    if done >= AUX_FEATURES_FRAMES:
        np.savez(AUX_FEATURES, index=np.concatenate(FEATURE_ROWS["index"]), x=np.concatenate(FEATURE_ROWS["x"]))
        print(f"[aux features] saved {done} frames to {AUX_FEATURES}", flush=True)
        os._exit(0)
    train_metrics.loss, train_metrics.grad_norm, train_metrics.lr = 0.0, 0.0, 0.0
    train_metrics.update_s = 0.0
    return train_metrics, {}


CALIBRATION = []


def round_13(x):
    import math
    candidates = [m * 10.0 ** e for e in range(-6, 7) for m in (1, 3)]
    return float(f"{min(candidates, key=lambda c: abs(math.log(c / x))):.3g}")


def update_policy_calibrate(train_metrics, policy, batch, optimizer, grad_clip_norm, accelerator, **kwargs):
    """No training: gradient norms on the vision encoder of the action loss and of the auxiliary
    loss, from the starting weights (design note D4: lambda gives the auxiliary term about half
    the action loss's vision gradient)."""
    import statistics
    inner = accelerator.unwrap_model(policy, keep_fp32_wrapper=True)
    inner.train()
    vision = [p for n, p in inner.named_parameters() if p.requires_grad and group(n) == "vision"]
    AUX.features.clear()
    with accelerator.autocast():
        loss, _ = policy.forward(batch)
        aux = AUX.error_sum(batch) / max(AUX.count(batch), 1.0)

    def norm(t):
        grads = torch.autograd.grad(t, vision, retain_graph=True, allow_unused=True)
        return float(torch.sqrt(sum((g.float() ** 2).sum() for g in grads if g is not None)))
    row = {"step": len(CALIBRATION) + 1, "index": batch["index"].view(-1).tolist(),
           "action_loss": loss.item(), "aux_loss": aux.item(),
           "vision_grad_norm_action": norm(loss), "vision_grad_norm_aux": norm(aux)}
    CALIBRATION.append(row)
    ga = statistics.median(r["vision_grad_norm_action"] for r in CALIBRATION)
    gx = statistics.median(r["vision_grad_norm_aux"] for r in CALIBRATION)
    report = {"batches": CALIBRATION, "median_vision_grad_norm_action": ga, "median_vision_grad_norm_aux": gx,
              "lambda_exact": 0.5 * ga / gx, "lambda": round_13(0.5 * ga / gx),
              "rule": "lambda = 0.5 x median action / median aux vision gradient norm, rounded to {1, 3} x 10^n"}
    Path(AUX_CALIBRATE).write_text(json.dumps(report, indent=1))
    print("[aux calibrate] " + json.dumps({k: v for k, v in row.items() if k != "index"}), flush=True)
    train_metrics.loss, train_metrics.grad_norm, train_metrics.lr = loss.item(), 0.0, 0.0
    train_metrics.update_s = 0.0
    return train_metrics, {}


# DataLoader workers start with "spawn" and re-import this file as __mp_main__.
if __name__ == "__main__":
    lt.make_policy = make_policy_fp32
    _update_policy = lt.update_policy
    if AUX_FEATURES:
        lt.update_policy = update_policy_features
        print(f"[fp32 launcher] aux features: {AUX_FEATURES_FRAMES} frames to {AUX_FEATURES}", flush=True)
    elif AUX_CALIBRATE:
        lt.update_policy = update_policy_calibrate
        print(f"[fp32 launcher] aux calibration to {AUX_CALIBRATE}", flush=True)
    elif ACCUM_CHECK:
        lt.update_policy = update_policy_checked
        print(f"[fp32 launcher] accumulation check: whole batch against micro-batches of {MICRO_BATCH}", flush=True)
    elif MICRO_BATCH or AUX_DIR:
        lt.update_policy = update_policy_accumulated
        print(f"[fp32 launcher] gradient accumulation: micro-batches of {MICRO_BATCH or 'the whole batch'}", flush=True)
    lt.update_policy = with_scaler(lt.update_policy)
    lt.save_checkpoint = save_checkpoint_with_scaler
    lt.load_training_state = load_training_state_with_scaler
    sys.exit(lt.main())
'''
# lerobot prints the step through format_big_number ("step:950", then "step:1K"), so from 1000 on
# the count comes from the tqdm prefix on the same line ("1000/20000 ["); after a resume that
# prefix restarts at 1, so below 1000 the step: field is used.
STEP_RE = re.compile(r'step:([\d.]+)([KMB]?)\s.*?loss:([\d.eE+-]+).*?updt_s:([\d.]+).*?data_s:([\d.]+)')
TQDM_RE = re.compile(r'(\d+)/\d+ \[')
MEM_RE = re.compile(r'mem_gb:([\d.]+)')
AUX_RE = re.compile(r'aux_loss:([\d.eE+-]+)')
SCALE_RE = re.compile(r'(?:scale now|loss scaler restored: scale) ([\d.eE+-]+)')
SUFFIX = {'': 1, 'K': 1_000, 'M': 1_000_000, 'B': 1_000_000_000}
OOM_MARKERS = ('OutOfMemoryError', 'CUDA out of memory')


def parse_step(line, m, offset=0):
    """offset: the step a resumed run started from, since the tqdm prefix counts from the resume."""
    if not m[2]:
        return int(m[1])
    t = TQDM_RE.search(line[:m.start()])
    return int(t[1]) + offset if t else int(float(m[1]) * SUFFIX[m[2]])


def log(msg):
    print(f'[driver {time.strftime("%H:%M:%S")}] {msg}', flush=True)


def find_dataset(input_root, scratch):
    for root_dir in (Path(input_root), Path(scratch) / 'unzipped'):
        for info in sorted(root_dir.rglob('meta/info.json')):
            root = info.parent.parent
            features = json.loads(info.read_text()).get('features', {})
            if all(k in features for k in CAMERA_RENAME) and (root / 'videos').is_dir():
                return root
        archives = sorted(Path(input_root).rglob('mycobot_tri_sort.zip'))
        if archives and root_dir == Path(input_root):
            log(f'extracting {archives[0]}')
            shutil.unpack_archive(archives[0], Path(scratch) / 'unzipped')
    sys.exit(f'No mycobot_tri_sort dataset (meta/info.json with top/right/left, or mycobot_tri_sort.zip) '
             f'under {input_root}. Attach the uploaded dataset as an input.')


def stage_dataset(cfg):
    src = find_dataset(cfg['input_root'], cfg['scratch'])
    dst = Path(cfg['data_root']) / REPO_ID
    if not (dst / 'meta' / 'info.json').exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns('images'))
    else:
        # A copy staged earlier in the same session keeps its data, but meta/ always follows the
        # source: Run C's aux_labels.npz and aux_head_init.safetensors may be newer than the copy.
        shutil.copytree(src / 'meta', dst / 'meta', dirs_exist_ok=True)
    info = json.loads((dst / 'meta' / 'info.json').read_text())
    log(f'dataset {src} -> {dst}: {info["total_episodes"]} episodes, {info["total_frames"]} frames, '
        f'fps {info["fps"]}')
    expected = {k: cfg[f'expect_{k}'] for k in ('episodes', 'frames') if cfg.get(f'expect_{k}')}
    found = {'episodes': info['total_episodes'], 'frames': info['total_frames']}
    if any(found[k] != v for k, v in expected.items()):
        sys.exit(f'Wrong dataset: {src} holds {found}, this run expects {expected}. Attach only the run\'s '
                 f'dataset; delete {dst} if it was staged from another one in this session.')
    return dst


def drop_incomplete_checkpoints(run_dir):
    """After lerobot-train has exited, delete checkpoints newer than `last`: lerobot moves `last`
    only once a save is complete, so a newer one was cut mid-write and must not be resumed from."""
    last = Path(run_dir) / 'checkpoints' / 'last'
    if not last.is_symlink():
        return
    for d in last.parent.iterdir():
        if d.is_dir() and not d.is_symlink() and d.name.isdigit() and int(d.name) > int(last.resolve().name):
            shutil.rmtree(d, ignore_errors=True)
            log(f'deleted {d.name}: written past `last`, incomplete')


# Every file lerobot writes in training_state; scheduler_state.json comes last. A session stopped
# by the time limit in the middle of a save leaves a checkpoint with only some of them (CPU test,
# 19/10: optimizer state written, parameter groups and scheduler not), which must not be resumed.
# The `last` symlink is not used: Kaggle may not keep symlinks in a session's output.
TRAINING_STATE_FILES = ('training_step.json', 'rng_state.safetensors', 'optimizer_state.safetensors',
                        'optimizer_param_groups.json', 'scheduler_state.json')


def complete_checkpoints(root):
    found = []
    for cfg_file in Path(root).rglob('checkpoints/*/pretrained_model/train_config.json'):
        ckpt = cfg_file.parent.parent
        if ckpt.is_symlink() or not ckpt.name.isdigit():
            continue
        if (all((ckpt / 'training_state' / f).is_file() for f in TRAINING_STATE_FILES)
                and any((ckpt / 'pretrained_model').glob('*.safetensors'))):
            found.append((int(ckpt.name), ckpt))
    return sorted(found)


def prune_checkpoints(run_dir, keep_every=0, kept_root=None):
    """Delete checkpoints older than the one `last` points to.

    lerobot moves `last` only after a save completes, and a checkpoint newer than `last` may be
    one still being written, so both are kept. Safe to call at any moment. With keep_every, the
    weights of every keep_every-th step move to kept_root/<step>/pretrained_model first (no
    optimizer state: 1.8 GB instead of ~3.5), so earlier checkpoints can be scored in Gazebo.
    """
    ckpt_root = Path(run_dir) / 'checkpoints'
    last = ckpt_root / 'last'
    if not last.is_symlink():
        return
    keep = int(last.resolve().name)
    for d in ckpt_root.iterdir():
        if d.is_dir() and not d.is_symlink() and d.name.isdigit() and int(d.name) < keep:
            if keep_every and int(d.name) % keep_every == 0 and (d / 'pretrained_model').is_dir():
                dst = Path(kept_root) / d.name / 'pretrained_model'
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(d / 'pretrained_model'), str(dst))
                    log(f'kept weights of step {d.name} in {dst}')
            shutil.rmtree(d, ignore_errors=True)
            log(f'pruned {d.name}')


class GpuMemSampler(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True)
        self.peak_mb = None
        self.stop = threading.Event()

    def run(self):
        if not shutil.which('nvidia-smi'):
            return
        while not self.stop.wait(1.0):
            try:
                out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits',
                                      '-i', os.environ.get('CUDA_VISIBLE_DEVICES', '0').split(',')[0]],
                                     capture_output=True, text=True, timeout=10).stdout.strip()
                self.peak_mb = max(self.peak_mb or 0, int(out.splitlines()[0]))
            except (ValueError, IndexError, subprocess.TimeoutExpired):
                pass


class Guards:
    """Run C stop conditions (cfg['guards'], fixed in probe_protocol.yaml before the run): the
    auxiliary loss above aux_loss_max on a step up to aux_loss_steps; more than skip_max updates
    skipped by the fp16 loss scaler among steps 1..skip_steps; the scaler below scale_min on a step
    after scale_after; a frozen head that differs from its warm start at a checkpoint."""

    def __init__(self, g, use_amp):
        self.g, self.reason = g, None
        self.scale = 65536.0 if use_amp else None  # GradScaler's initial scale; a resume restores its own
        self.pending, self.skipped = 0, 0

    def line(self, line, step):
        if 'update skipped by the fp16 loss scaler' in line:
            self.pending += 1
        s = SCALE_RE.search(line)
        if s:
            self.scale = float(s[1])
        if 'aux head frozen: identical' in line and ' False,' in line:
            self.reason = 'frozen aux head differs from aux_head_init.safetensors at a checkpoint'
        if step is None:
            return self.reason
        g = self.g
        if step <= g['skip_steps']:
            self.skipped += self.pending
        self.pending = 0
        a = AUX_RE.search(line)
        if a and step <= g['aux_loss_steps'] and float(a[1]) > g['aux_loss_max']:
            self.reason = f"aux_loss {float(a[1]):.4g} > {g['aux_loss_max']} at step {step}"
        elif self.skipped > g['skip_max']:
            self.reason = f"{self.skipped} updates skipped by the fp16 loss scaler in steps 1-{g['skip_steps']}"
        elif step > g['scale_after'] and self.scale is not None and self.scale < g['scale_min']:
            self.reason = f"fp16 loss scaler at {self.scale:g} < {g['scale_min']} at step {step}"
        return self.reason


def run_lerobot(cfg, args, log_path, deadline=None, prune_dir=None, stop_after_step=None, guards=None,
                step_offset=0):
    """Stream lerobot-train output; stop it cleanly at the deadline, once a step past stop_after_step
    is logged (resume check), or on a guard (Guards). Returns (rc, steps, oom, peak_mb, stopped);
    a guard stop is in guards.reason."""
    keep = dict(keep_every=cfg.get('keep_every', 0), kept_root=Path(cfg['working']) / 'kept')
    if cfg.get('fp32_weights', True):
        launcher = Path(cfg['scratch']) / 'lerobot_train_fp32.py'
        launcher.write_text(FP32_LAUNCHER)
        cmd = [str(Path(cfg['venv']) / 'bin' / 'python'), str(launcher)] + args
    else:
        cmd = [str(Path(cfg['venv']) / 'bin' / 'lerobot-train')] + args
    env = dict(os.environ, HF_HOME=cfg['hf_home'], WANDB_MODE='disabled', PYTHONUNBUFFERED='1',
               SMOLVLA_FREEZE=cfg.get('freeze') or '', SMOLVLA_MICRO_BATCH=str(cfg.get('micro_batch') or ''),
               SMOLVLA_ACCUM_CHECK=cfg.get('accum_check') or '',
               SMOLVLA_AUX_DIR=cfg.get('aux_dir') or '', SMOLVLA_AUX_LAMBDA=str(cfg.get('aux_lambda') or ''),
               SMOLVLA_LR_VISION=str(cfg.get('lr_vision') or ''), SMOLVLA_LR_HEAD=str(cfg.get('lr_head') or ''),
               SMOLVLA_AUX_FEATURES=cfg.get('aux_features') or '',
               SMOLVLA_AUX_FEATURES_FRAMES=str(cfg.get('aux_features_frames') or ''),
               SMOLVLA_AUX_CALIBRATE=cfg.get('aux_calibrate') or '',
               SMOLVLA_AUX_HEAD_FROZEN='1' if cfg.get('aux_head_frozen') else '')
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log('run: ' + ' '.join(cmd))
    sampler = GpuMemSampler()
    sampler.start()
    steps, oom, stopped = [], False, False
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
                            start_new_session=True)
    watchdog = None
    if deadline:
        def stop_at_deadline():
            nonlocal stopped
            while proc.poll() is None:
                if time.time() > deadline:
                    stopped = True
                    log('session time limit reached: stopping training (resume from the last checkpoint)')
                    os.killpg(proc.pid, signal.SIGINT)
                    try:
                        proc.wait(timeout=180)
                    except subprocess.TimeoutExpired:
                        os.killpg(proc.pid, signal.SIGKILL)
                    return
                time.sleep(10)
        watchdog = threading.Thread(target=stop_at_deadline, daemon=True)
        watchdog.start()
    # Pruning at every step line and every "Checkpoint" line keeps at most two checkpoints on disk:
    # the one `last` points to and the one being written.
    with open(log_path, 'a') as f:
        for line in proc.stdout:
            f.write(line)
            if any(m in line for m in OOM_MARKERS):
                oom = True
            m = STEP_RE.search(line)
            step = parse_step(line, m, step_offset) if m else None
            if m:
                mem = MEM_RE.search(line)
                steps.append({'step': step, 'loss': float(m[3]), 'updt_s': float(m[4]), 'data_s': float(m[5]),
                              'mem_gb': float(mem[1]) if mem else None})
                if stop_after_step and step > stop_after_step and not stopped:
                    stopped = True
                    log(f'step {step} logged: stopping training as a session cut would (resume check)')
                    os.killpg(proc.pid, signal.SIGINT)
            if guards and guards.reason is None and guards.line(line, step):
                stopped = True
                log(f'GUARD STOP: {guards.reason}; stopping training')
                os.killpg(proc.pid, signal.SIGINT)
            if prune_dir and (m or 'Checkpoint policy after step' in line):
                prune_checkpoints(prune_dir, **keep)
            if m or 'Checkpoint' in line or '[fp32 launcher]' in line or '[accum check]' in line or (
                    '[aux calibrate]' in line or ('[aux features]' in line and 'saved' in line)) or (
                    ('Traceback' in line or 'Error' in line) and 'torchcodec' not in line
                    and 'libnppicc' not in line and 'Could not load this library' not in line):
                print(line.rstrip()[-300:], flush=True)
    rc = proc.wait()
    sampler.stop.set()
    if prune_dir:
        prune_checkpoints(prune_dir, **keep)
    return rc, steps, oom, sampler.peak_mb, stopped


def common_args(cfg, data_dir, batch_size, steps, use_amp, schedule_steps=None):
    schedule_steps = schedule_steps or steps
    return [
        '--policy.path=lerobot/smolvla_base',
        f'--policy.device={cfg["device"]}',
        '--policy.push_to_hub=false',
        f'--policy.use_amp={str(use_amp).lower()}',
        f'--policy.scheduler_decay_steps={schedule_steps}',
        f'--policy.scheduler_warmup_steps={min(1000, max(1, schedule_steps // 20))}',
        f'--dataset.repo_id={REPO_ID}',
        f'--dataset.root={data_dir}',
        '--rename_map=' + json.dumps(CAMERA_RENAME, separators=(',', ':')),
        f'--batch_size={batch_size}',
        f'--steps={steps}',
        f'--num_workers={cfg["num_workers"]}',
        '--wandb.enable=false',
        f'--seed={cfg["seed"]}',
    ] + ([f'--dataset.video_backend={cfg["video_backend"]}'] if cfg.get('video_backend') else []) + (
        ['--policy.freeze_vision_encoder=false', '--policy.train_expert_only=false']
        if cfg.get('freeze') == 'runC' else [])


def benchmark(cfg, data_dir):
    work = Path(cfg['working'])
    rows = []
    for use_amp in cfg['bench_amp']:
        for bs in cfg['bench_batch_sizes']:
            out = Path(cfg['scratch']) / f'bench_bs{bs}_amp{int(use_amp)}'
            shutil.rmtree(out, ignore_errors=True)
            args = common_args(cfg, data_dir, bs, cfg['bench_steps'], use_amp) + [
                '--log_freq=1', '--save_checkpoint=false', f'--output_dir={out}']
            t0 = time.time()
            rc, steps, oom, peak, _ = run_lerobot(cfg, args, work / 'logs' / f'bench_bs{bs}_amp{int(use_amp)}.log')
            shutil.rmtree(out, ignore_errors=True)
            steady = [s for s in steps if s['step'] > cfg['bench_warmup_steps']]
            row = {'batch_size': bs, 'amp': use_amp, 'rc': rc, 'oom': oom, 'peak_gpu_mb': peak,
                   'wall_s': round(time.time() - t0)}
            if steady and rc == 0:
                updt = sum(s['updt_s'] for s in steady) / len(steady)
                data = sum(s['data_s'] for s in steady) / len(steady)
                row.update(s_per_step=round(updt + data, 3), updt_s=round(updt, 3), data_s=round(data, 3),
                           samples_per_s=round(bs / (updt + data), 2),
                           mem_gb=max((s['mem_gb'] for s in steps if s['mem_gb'] is not None), default=None))
            rows.append(row)
            log(f'benchmark {row}')
            if oom or rc != 0:
                break
    (work / 'benchmark.json').write_text(json.dumps(rows, indent=2))
    ok = [r for r in rows if 's_per_step' in r]
    print('\n batch  amp    s/step  data_s  samples/s  mem_gb  peak GPU MB   h for train_steps   steps in 11 h')
    for r in ok:
        hours = r['s_per_step'] * cfg['train_steps'] / 3600
        print(f" {r['batch_size']:5d}  {str(r['amp']):5s}  {r['s_per_step']:6.2f}  {r['data_s']:6.2f}  "
              f"{r['samples_per_s']:9.2f}  {str(r['mem_gb']):>6s}  {str(r['peak_gpu_mb']):>11s}   {hours:17.1f}   "
              f"{int(11 * 3600 / r['s_per_step']):13d}")
    for r in rows:
        if 's_per_step' not in r:
            print(f" {r['batch_size']:5d}  {str(r['amp']):5s}  {'out of memory' if r['oom'] else 'failed rc=' + str(r['rc'])}"
                  f" (log: logs/bench_bs{r['batch_size']}_amp{int(r['amp'])}.log)")
    if ok:
        best = max(ok, key=lambda r: r['samples_per_s'])
        print(f"\nFastest: batch {best['batch_size']}, amp {best['amp']}: {best['samples_per_s']} samples/s. "
              f"train_steps={cfg['train_steps']} at that batch would take "
              f"{best['s_per_step'] * cfg['train_steps'] / 3600:.1f} h of GPU.")
        if best['data_s'] > 0.3 * best['s_per_step']:
            print('Note: data loading is over 30 % of each step; video decoding on the CPU is limiting.')


def train(cfg, data_dir):
    work = Path(cfg['working'])
    run_dir = work / 'run'
    # A Kaggle session's output holds only what it wrote: carry the kept weights of earlier
    # sessions (attached as input) forward, so the last session's output has them all.
    for kept in sorted(Path(cfg['input_root']).rglob('kept/*/pretrained_model')):
        dst = work / 'kept' / kept.parent.name / 'pretrained_model'
        if kept.parent.name.isdigit() and not dst.exists():
            shutil.copytree(kept, dst)
            log(f'carried kept weights of step {kept.parent.name} forward')
    previous = complete_checkpoints(cfg['input_root'])
    if previous and previous[-1][0] >= cfg['train_steps']:
        log(f'TRAINING COMPLETE already: the attached input holds step {previous[-1][0]} ({previous[-1][1]})')
        return
    if previous:
        step, ckpt = previous[-1]
        log(f'resuming from step {step}: {ckpt}')
        # A session cut right after a kept step leaves that checkpoint only in the input,
        # where pruning never sees it.
        dst = work / 'kept' / ckpt.name / 'pretrained_model'
        if cfg.get('keep_every') and step % cfg['keep_every'] == 0 and not dst.exists():
            shutil.copytree(ckpt / 'pretrained_model', dst)
            log(f'kept weights of step {ckpt.name} (resumed from) in {dst}')
        args = [f'--config_path={ckpt / "pretrained_model" / "train_config.json"}', '--resume=true']
        step_offset = step
    else:
        if run_dir.exists() and complete_checkpoints(run_dir):
            sys.exit(f'{run_dir} already holds checkpoints from this session; attach it as input to resume, '
                     'or delete it to start over.')
        shutil.rmtree(run_dir, ignore_errors=True)
        log('no previous checkpoint attached: fresh run from lerobot/smolvla_base')
        step_offset = 0
        args = common_args(cfg, data_dir, cfg['train_batch_size'], cfg['train_steps'], cfg['train_amp']) + [
            f'--save_freq={cfg["save_freq"]}', f'--log_freq={cfg["log_freq"]}', f'--output_dir={run_dir}']
    deadline = time.time() + cfg['session_hours'] * 3600
    guards = Guards(cfg['guards'], cfg['train_amp']) if cfg.get('guards') else None
    rc, steps, oom, peak, stopped = run_lerobot(cfg, args, work / 'logs' / f'train_{time.strftime("%Y%m%d_%H%M%S")}.log',
                                                deadline=deadline, prune_dir=run_dir,
                                                stop_after_step=cfg.get('stop_after_step'), guards=guards,
                                                step_offset=step_offset)
    drop_incomplete_checkpoints(run_dir)
    done = complete_checkpoints(run_dir)
    guard_stop = guards.reason if guards else None
    summary = {'rc': rc, 'oom': oom, 'stopped_at_time_limit': stopped and not guard_stop, 'guard_stop': guard_stop,
               'updates_skipped_steps_1_100': guards.skipped if guards else None,
               'fp16_scale_last': guards.scale if guards else None, 'peak_gpu_mb': peak,
               'last_logged_step': steps[-1]['step'] if steps else None,
               'last_logged_loss': steps[-1]['loss'] if steps else None,
               'checkpoint': str(done[-1][1]) if done else None,
               'target_steps': cfg['train_steps']}
    (work / 'train_summary.json').write_text(json.dumps(summary, indent=2))
    log(f'summary {summary}')
    if guard_stop:
        sys.exit(f'GUARD STOP: {guard_stop}. Do not resume this run: the stop condition in probe_protocol.yaml '
                 'applies.')
    if not done:
        sys.exit('No complete checkpoint was written in this session.')
    if done[-1][0] >= cfg['train_steps']:
        log(f'TRAINING COMPLETE. Download {done[-1][1] / "pretrained_model"}')
    else:
        log(f'Stopped at step {done[-1][0]} of {cfg["train_steps"]}. Next session: attach this output as input '
            'and run again in train mode; it resumes from this checkpoint.')


# Accumulation check pass rule, fixed before the first GPU run (19/10/2026): on every step where
# neither pass was skipped by the fp16 loss scaler (at least 3 such steps), the accumulated pass
# against the whole-batch pass: loss, pre-clip gradient norm and gradients (overall and per
# group) within 1 %, weight update (overall and per group) within 5 %. The update is looser
# because Adam's first steps divide each gradient by its own magnitude, so entries near zero
# can flip sign on rounding alone.
CHECK_LIMITS = {'loss_rel': 0.01, 'grad_norm_rel': 0.01, 'grads_rel': 0.01, 'update_rel': 0.05}
CHECK_MIN_STEPS = 3


def accum_check(cfg, data_dir):
    work = Path(cfg['working'])
    out = Path(cfg['scratch']) / 'accum_check_run'
    report = work / 'accum_check.json'
    shutil.rmtree(out, ignore_errors=True)
    report.unlink(missing_ok=True)
    args = common_args(cfg, data_dir, cfg['check_batch'], cfg['check_steps'], cfg['train_amp'],
                       schedule_steps=cfg['train_steps']) + [
        '--log_freq=1', '--save_checkpoint=false', f'--output_dir={out}']
    rc, _, oom, peak, _ = run_lerobot(dict(cfg, micro_batch=cfg['check_micro'], accum_check=str(report)), args,
                                      work / 'logs' / 'accum_check.log')
    shutil.rmtree(out, ignore_errors=True)
    rows = json.loads(report.read_text()) if report.exists() else []
    judged, failures = 0, []
    print(f"\n accumulation check: batch {cfg['check_batch']} against {cfg['check_batch'] // cfg['check_micro']} x "
          f"{cfg['check_micro']}; rc {rc}, oom {oom}, nvidia-smi peak {peak} MB")
    print(' step   loss   grad_norm   floor grads  floor update   loss_rel  grad_norm_rel  grads_rel  update_rel   per group grads / update (floor update)')
    for r in rows:
        a, f = r['accumulated_vs_whole'], r['whole_again_vs_whole']
        if a.get('skipped'):
            print(f" {r['step']:4d}   skipped by the fp16 loss scaler")
            continue
        judged += 1
        groups = ', '.join(f"{k[10:]} {a[k]:.1e}/{a['update_rel_' + k[10:]]:.1e} "
                           f"({f.get('update_rel_' + k[10:], float('nan')):.1e})"
                           for k in sorted(a) if k.startswith('grads_rel_'))
        print(f" {r['step']:4d}  {r['loss']:.4f}  {r['grad_norm']:9.3f}   {f.get('grads_rel', float('nan')):.1e}      "
              f"{f.get('update_rel', float('nan')):.1e}      "
              f"{a['loss_rel']:.1e}    {a['grad_norm_rel']:.1e}       {a['grads_rel']:.1e}    {a['update_rel']:.1e}     {groups}")
        for k, v in a.items():
            limit = CHECK_LIMITS.get(next((b for b in ('grads_rel', 'update_rel') if k.startswith(b)), k))
            if limit is not None and not v <= limit:
                failures.append(f"step {r['step']}: {k} {v:.2e} > {limit}")
        if r['missing_grads']:
            failures.append(f"step {r['step']}: trainable tensors without a gradient, e.g. {r['missing_grads']}")
    if judged < CHECK_MIN_STEPS:
        failures.append(f'only {judged} steps judged (need {CHECK_MIN_STEPS})')
    verdict = {'pass': not failures and rc == 0, 'judged_steps': judged, 'failures': failures, 'rc': rc, 'oom': oom,
               'peak_gpu_mb': peak, 'limits': CHECK_LIMITS}
    (work / 'accum_check_verdict.json').write_text(json.dumps(verdict, indent=2))
    print('\nACCUMULATION CHECK ' + ('PASS' if verdict['pass'] else 'FAIL') + ''.join('\n  ' + x for x in failures))


def logged_steps(log_path):
    rows = {}
    for line in Path(log_path).read_text().splitlines():
        m = STEP_RE.search(line)
        if m:
            rows[parse_step(line, m)] = float(m[3])
    return rows


def resume_check(cfg, data_dir):
    """The same run twice through train(): once in one go, once stopped right after step
    resume_stop_step (SIGINT, as the session time limit does; the checkpoint before it is complete)
    and resumed from its output attached as input. Compares the final checkpoints (step, scheduler,
    weights, optimizer state) and the per-step losses."""
    import torch
    from safetensors.torch import load_file
    base = Path(cfg['scratch']) / 'resume_check'
    shutil.rmtree(base, ignore_errors=True)
    no_input = base / 'no_input'
    no_input.mkdir(parents=True)
    common = dict(cfg, mode='train', train_steps=cfg['resume_steps'], save_freq=cfg['resume_save_freq'],
                  log_freq=1, keep_every=0)
    t0 = time.time()
    one = dict(common, working=str(base / 'one_go'), input_root=str(no_input), session_hours=24)
    train(one, data_dir)
    one_go_s = time.time() - t0
    # As on Kaggle: both sessions write to the same working directory (the checkpoint's
    # train_config.json names it as output_dir), and the first one's output comes back as input.
    session = base / 'session'
    first = dict(common, working=str(session), input_root=str(no_input), session_hours=24,
                 stop_after_step=cfg['resume_stop_step'])
    try:
        train(first, data_dir)
    except SystemExit as e:
        log(f'first session ended: {e}')
    first['working'] = str(base / 'first_output')
    shutil.move(str(session), first['working'])
    stopped_at = complete_checkpoints(Path(first['working']) / 'run')
    second = dict(common, working=str(session), input_root=first['working'], session_hours=24)
    try:
        train(second, data_dir)
    except SystemExit as e:
        result = {'pass': False, 'failure': f'second session: {e}', 'resumed_from_step': stopped_at[-1][0] if stopped_at else None}
        (Path(cfg['working']) / 'resume_check.json').write_text(json.dumps(result, indent=2))
        print('\nRESUME CHECK ' + json.dumps(result, indent=2))
        return

    def final(work):
        step, ckpt = complete_checkpoints(Path(work) / 'run')[-1]
        state = ckpt / 'training_state'
        scaler = state / 'grad_scaler.json'
        return {'step': step, 'weights': load_file(ckpt / 'pretrained_model' / 'model.safetensors'),
                'optimizer': load_file(state / 'optimizer_state.safetensors'),
                'scheduler': json.loads((state / 'scheduler_state.json').read_text()),
                'scaler': json.loads(scaler.read_text()) if scaler.is_file() else None}

    def skipped(*works):
        return sum(p.read_text().count('update skipped by the fp16 loss scaler')
                   for w in works for p in (Path(w) / 'logs').glob('train_*.log'))
    a, b = final(one['working']), final(second['working'])

    def rel_diff(x, y):
        keys = sorted(set(x) & set(y))
        num = sum(float(((x[k].double() - y[k].double()) ** 2).sum()) for k in keys)
        den = sum(float((x[k].double() ** 2).sum()) for k in keys)
        return (num / den) ** 0.5, len(keys), len(set(x) ^ set(y))
    w_rel, w_n, w_extra = rel_diff(a['weights'], b['weights'])
    o_rel, o_n, o_extra = rel_diff(a['optimizer'], b['optimizer'])
    la, lb = logged_steps(newest_log(one['working'])), {}
    for p in sorted((Path(first['working']) / 'logs').glob('train_*.log')) + sorted((Path(second['working']) / 'logs').glob('train_*.log')):
        lb.update(logged_steps(p))
    resumed_from = stopped_at[-1][0] if stopped_at else None
    # The data order after a resume is exact, but recreating the data loader takes one draw from
    # the restored random state, so the flow-matching noise and times differ from the one-go run
    # (traced on CPU, 19/10): losses are compared as means over the steps after the resume.
    after = [s for s in sorted(la) if resumed_from is not None and s > resumed_from and s in lb]
    xa, xb = [la[s] for s in after], [lb[s] for s in after]

    def mean_sd(x):
        m = sum(x) / len(x)
        return m, (sum((v - m) ** 2 for v in x) / max(1, len(x) - 1)) ** 0.5
    (ma, sa), (mb, sb) = (mean_sd(xa), mean_sd(xb)) if after else ((None, None), (None, None))
    se = ((sa ** 2 + sb ** 2) / len(after)) ** 0.5 if after else None
    missing = [s for s in range(1, cfg['resume_steps'] + 1) if s not in lb]
    no_jump = bool(after) and abs(mb - ma) <= 2 * se
    result = {'one_go_s': round(one_go_s), 'resumed_from_step': resumed_from,
              'final_step': [a['step'], b['step']], 'scheduler_equal': a['scheduler'] == b['scheduler'],
              'scheduler_one_go': a['scheduler'], 'scheduler_resumed': b['scheduler'],
              'fp16_scaler_one_go': a['scaler'], 'fp16_scaler_resumed': b['scaler'],
              'fp16_scaler_restored': 'fp16 loss scaler restored' in ''.join(
                  p.read_text() for p in (Path(second['working']) / 'logs').glob('train_*.log')),
              'updates_skipped': {'one_go': skipped(one['working']), 'first_session': skipped(first['working']),
                                  'second_session': skipped(second['working'])},
              'weights_rel_diff': w_rel, 'weights_tensors': w_n, 'weights_tensors_unmatched': w_extra,
              'optimizer_rel_diff': o_rel, 'optimizer_tensors_unmatched': o_extra,
              'steps_logged_one_go': len(la), 'steps_logged_resumed': len(lb),
              'missing_steps_resumed': missing[:20],
              'loss_mean_after_resume': {'steps': len(after), 'one_go': ma, 'resumed': mb, 'sd_one_go': sa,
                                         'sd_resumed': sb, 'difference': None if not after else mb - ma,
                                         'two_standard_errors': None if not after else 2 * se},
              'pass': (resumed_from is not None and a['step'] == b['step'] == cfg['resume_steps']
                       and a['scheduler'] == b['scheduler'] and not missing and no_jump),
              'losses_around_resume': {s: [la.get(s), lb.get(s)] for s in range((resumed_from or 0) - 3, (resumed_from or 0) + 6)
                                       if s in la or s in lb}}
    (Path(cfg['working']) / 'resume_check.json').write_text(json.dumps(result, indent=2))
    print('\nRESUME CHECK ' + json.dumps(result, indent=2))


def aux_features(cfg, data_dir):
    """Frozen smolvla_base connector features of aux_features_frames training frames, through the
    training data path (design note D4 warm start); fitted by scripts/aux_warm_start.py."""
    work = Path(cfg['working'])
    out = Path(cfg['scratch']) / 'aux_features_run'
    shutil.rmtree(out, ignore_errors=True)
    batch = cfg['aux_features_batch']
    steps = -(-cfg['aux_features_frames'] // batch) + 1
    target = work / 'aux_features.npz'
    args = common_args(cfg, data_dir, batch, steps, cfg['train_amp'], schedule_steps=cfg['train_steps']) + [
        '--log_freq=1', '--save_checkpoint=false', f'--output_dir={out}']
    rc, *_ = run_lerobot(dict(cfg, aux_dir=None, aux_features=str(target)), args, work / 'logs' / 'aux_features.log')
    shutil.rmtree(out, ignore_errors=True)
    log(f'aux features rc {rc}: {target} {"written" if target.exists() else "MISSING"}')


def aux_calibrate(cfg, data_dir):
    """Gradient norms on the vision encoder of both losses at the starting weights, with the warm-
    started head (design note D4); the suggested lambda is in aux_calibration.json. The batches are
    drawn with seed + 1: with the same seed the sampler would hand over the very frames the warm
    start was fitted on, where the head is nearly exact and its gradient tiny (CPU test, 19/10). The
    share of calibration frames that were in the warm-start sample is reported; it must be ~0."""
    work = Path(cfg['working'])
    out = Path(cfg['scratch']) / 'aux_calibrate_run'
    shutil.rmtree(out, ignore_errors=True)
    report = work / 'aux_calibration.json'
    calib = dict(cfg, aux_calibrate=str(report), seed=cfg['seed'] + 1)
    args = common_args(calib, data_dir, cfg['aux_calibrate_batch'], cfg['aux_calibrate_batches'], cfg['train_amp'],
                       schedule_steps=cfg['train_steps']) + [
        '--log_freq=1', '--save_checkpoint=false', f'--output_dir={out}']
    rc, *_ = run_lerobot(calib, args, work / 'logs' / 'aux_calibrate.log')
    shutil.rmtree(out, ignore_errors=True)
    if report.exists():
        r = json.loads(report.read_text())
        used = [i for b in r['batches'] for i in b['index']]
        fitted = Path(cfg['aux_fitted_index']) if cfg.get('aux_fitted_index') else work / 'aux_features.npz'
        if fitted.exists():
            import zipfile
            with zipfile.ZipFile(fitted) as z, z.open('index.npy') as f:
                header = f.read(128)
                fitted_index = set(int(v) for v in _npy_int_array(header + f.read()))
            r['share_in_warm_start_sample'] = sum(i in fitted_index for i in used) / len(used)
            report.write_text(json.dumps(r, indent=1))
        log(f"aux calibration rc {rc}: {len(r['batches'])} batches, lambda {r['lambda']} (exact {r['lambda_exact']:.3g}), "
            f"share of its frames in the warm-start sample {r.get('share_in_warm_start_sample', 'unknown')}")


def _npy_int_array(raw):
    """An integer .npy array, standard library only (the driver runs outside the venv)."""
    import ast
    import struct
    major = raw[6]
    hlen = struct.unpack('<H', raw[8:10])[0] if major == 1 else struct.unpack('<I', raw[8:12])[0]
    start = 10 if major == 1 else 12
    header = ast.literal_eval(raw[start:start + hlen].decode('latin1'))
    code = {'<i8': 'q', '<i4': 'i', '<u8': 'Q', '<u4': 'I'}[header['descr']]
    data = raw[start + hlen:]
    return struct.unpack(f'<{len(data) // struct.calcsize(code)}{code}', data)


def newest_log(work):
    return sorted((Path(work) / 'logs').glob('train_*.log'))[-1]


def main():
    cfg = json.loads(Path(sys.argv[1]).read_text())
    for k in ('working', 'scratch', 'data_root', 'hf_home'):
        Path(cfg[k]).mkdir(parents=True, exist_ok=True)
    data_dir = stage_dataset(cfg)
    if cfg.get('aux'):
        cfg['aux_dir'] = str(data_dir / 'meta')
    if cfg['mode'] == 'benchmark':
        benchmark(cfg, data_dir)
    elif cfg['mode'] == 'train':
        train(cfg, data_dir)
    elif cfg['mode'] == 'accumcheck':
        accum_check(cfg, data_dir)
    elif cfg['mode'] == 'resumecheck':
        resume_check(cfg, data_dir)
    elif cfg['mode'] == 'auxfeatures':
        aux_features(cfg, data_dir)
    elif cfg['mode'] == 'auxcalibrate':
        aux_calibrate(cfg, data_dir)
    else:
        sys.exit(f'unknown mode {cfg["mode"]!r}: use "benchmark", "train", "accumcheck", "resumecheck", '
                 '"auxfeatures" or "auxcalibrate"')


if __name__ == '__main__':
    main()

In [ ]:
# ---- 4. Run ------------------------------------------------------------------
import json, subprocess, sys
from pathlib import Path

Path(WORKING).mkdir(parents=True, exist_ok=True)
cfg = {
    "mode": MODE, "input_root": INPUT_ROOT, "working": WORKING,
    "scratch": LOCAL + "/scratch", "data_root": LOCAL + "/data", "hf_home": LOCAL + "/hf",
    "venv": VENV, "device": "cuda", "num_workers": NUM_WORKERS, "seed": SEED,
    "train_steps": TRAIN_STEPS, "train_batch_size": TRAIN_BATCH_SIZE, "micro_batch": MICRO_BATCH,
    "train_amp": TRAIN_AMP, "save_freq": SAVE_FREQ, "log_freq": LOG_FREQ, "session_hours": SESSION_HOURS,
    "video_backend": VIDEO_BACKEND, "fp32_weights": FP32_WEIGHTS, "keep_every": KEEP_EVERY,
    "freeze": FREEZE, "lr_vision": LR_VISION, "aux": True, "aux_lambda": AUX_LAMBDA,
    "aux_head_frozen": AUX_HEAD_FROZEN, "guards": GUARDS,
    "expect_episodes": EXPECT_EPISODES, "expect_frames": EXPECT_FRAMES,
}
Path(WORKING, "config.json").write_text(json.dumps(cfg, indent=2))
p = subprocess.Popen([sys.executable, "-u", "/tmp/smolvla_kaggle_driver.py", f"{WORKING}/config.json"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
rc = p.wait()
print(f"--- driver exited with {rc}")

In [ ]:
# ---- 5. What this session produced, and the frozen head's check -----------------
import json, subprocess
print(subprocess.run(f"cd {WORKING} && du -sh * 2>/dev/null; ls -l run/checkpoints kept 2>/dev/null; "
                     "cat train_summary.json 2>/dev/null", shell=True, capture_output=True, text=True).stdout)
ckpts = sorted(Path(WORKING, "run", "checkpoints").glob("[0-9]*/pretrained_model/aux_head.safetensors"))
check = f'''
import json, sys
import torch
from safetensors.torch import load_file
a, b = load_file(sys.argv[1]), load_file(sys.argv[2])
same = a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a)
print(json.dumps({{"checkpoint": sys.argv[1], "identical_to_init": same,
                  "weight_sum": float(a["weight"].double().sum()),
                  "max_abs_diff": max(float((a[k] - b[k]).abs().max()) for k in a if k in b)}}))
'''
if ckpts:
    out = subprocess.run([f"{VENV}/bin/python", "-c", check, str(ckpts[-1]), str(META / "aux_head_init.safetensors")],
                         capture_output=True, text=True)
    head = json.loads(out.stdout.strip().splitlines()[-1]) if out.returncode == 0 else {"error": out.stderr[-2000:]}
    head["pass"] = bool(head.get("identical_to_init")) and abs(head.get("weight_sum", 1e9) - AUX_INIT_WEIGHT_SUM) < 1e-6
    Path(WORKING, "runc_final_head.json").write_text(json.dumps(head, indent=1))
    print("HEAD CHECK", "PASS" if head["pass"] else "FAIL", head)
else:
    print("HEAD CHECK: no checkpoint in this session's output")
assert rc == 0, f"driver exited with {rc}; see {WORKING}/logs/ and train_summary.json"